# Preparación del corpus

Una vez se ha llevado a cabo el análisis exploratorio del corpus, es necesario
transformarlo de tal manera que el LLM sea capaz de entenderlo.

Para ello, se llevan a cabo cuatro acciones principales:

1. Carga del corpus: Se carga el corpus y se separa cada ejemplo con el token
   `<|endoftext|>` para que el LLM sepa donde termina cada historia.
2. División del corpus en tokens: Se utiliza la librería tiktoken, que
   implementa el algoritmo byte-pair encoding, para convertir el texto a token
   IDs.
3. Sliding window: Se crea un DataLoader capaz de proveer los datos de
   entrenamiento al LLM utilizando la técnica sliding window. Esta técnica
   limita la cantidad de texto que el LLM tiene que "leer" durante el
   entrenamiento, mejorando su eficiencia.
4. Transformación a token embeddings: TODO

## Carga del corpus

Al igual que se hizo en el análisis exploratorio del corpus, se utiliza la
librería `datasets` para cargar el corpus fácilmente.

Se concatenan los ejemplos del corpus añadiendo el token `<|endoftext|>` para
que el LLM sepa donde termina cada historia.

In [1]:
from datasets import load_dataset
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
ds = load_dataset("roneneldan/TinyStories")

In [5]:
NUM_EXAMPLES = 200
all_text = "<|endoftext|>".join(ds["train"]["text"][:NUM_EXAMPLES])

In [6]:
all_text[675:725]

'hared and worked together.<|endoftext|>Once upon a'

In [18]:
ds["train"]["text"][0][-30:]

'ad shared and worked together.'

In [19]:
ds["train"]["text"][1][:30]

'Once upon a time, there was a '

## Tokenización y sliding window

Se crea el DataSet y el DataLoader para proveer los datos al modelo utilizando
la técnica sliding window.

In [7]:
class TinyDataset(Dataset):
    def __init__(self, all_text, window_size):
        self.window_size = window_size

        tokenizer = tiktoken.get_encoding("gpt2")
        self.token_ids = tokenizer.encode(all_text, allowed_special={"<|endoftext|>"})

    def __len__(self):
        return len(self.token_ids) // self.window_size
    
    def __getitem__(self, index):
        i = index * self.window_size
        x_chunk = self.token_ids[i : i + self.window_size]
        y_chunk = self.token_ids[i + 1 : i + self.window_size + 1]
        return torch.tensor(x_chunk), torch.tensor(y_chunk)

In [8]:
def create_dataloader(all_text, batch_size, window_size, shuffle=True, drop_last=True, num_workers = 0):
    dataset = TinyDataset(all_text, window_size)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [9]:
batch_size = 8
window_size = 4

dataloader = create_dataloader(all_text, batch_size, window_size, shuffle=False, drop_last=False)

In [10]:
dl_iter = iter(dataloader)
x, y = next(dl_iter)
print(f"x:\n{x}")
print(f"y:\n{y}")

x:
tensor([[ 3198,  1110,    11,   257],
        [ 1310,  2576,  3706, 20037],
        [ 1043,   257, 17598,   287],
        [  607,  2119,    13,  1375],
        [ 2993,   340,   373,  2408],
        [  284,   711,   351,   340],
        [  780,   340,   373,  7786],
        [   13, 20037,  2227,   284]])
y:
tensor([[ 1110,    11,   257,  1310],
        [ 2576,  3706, 20037,  1043],
        [  257, 17598,   287,   607],
        [ 2119,    13,  1375,  2993],
        [  340,   373,  2408,   284],
        [  711,   351,   340,   780],
        [  340,   373,  7786,    13],
        [20037,  2227,   284,  2648]])


In [11]:
x, y = next(dl_iter)
print(f"x:\n{x}")
print(f"y:\n{y}")

x:
tensor([[ 2648,   262, 17598,   351],
        [  607,  1995,    11,   523],
        [  673,   714, 34249,   257],
        [ 4936,   319,   607, 10147],
        [   13,   198,   198,    43],
        [  813,  1816,   284,   607],
        [ 1995,   290,   531,    11],
        [  366, 29252,    11,   314]])
y:
tensor([[  262, 17598,   351,   607],
        [ 1995,    11,   523,   673],
        [  714, 34249,   257,  4936],
        [  319,   607, 10147,    13],
        [  198,   198,    43,   813],
        [ 1816,   284,   607,  1995],
        [  290,   531,    11,   366],
        [29252,    11,   314,  1043]])


## Transformación a token embeddings

TODO -> añadir una descripción al principio también